# Setup and Overview

## Goal

Prepare the core notebook environment and introduce the geoparsing workflow.

## What you will do

- Confirm that the notebook is running from the project.
- Install or verify the lightweight core packages.
- Load sample texts and test service reachability.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Python version:", platform.python_version())

## Step 1: Geoparsing workflow

Raw text -> Place-name extraction -> Toponym resolution -> Coordinates -> Map visualization -> Applications

## Step 2: Check folders

In [ ]:
required = ['notebooks', 'src', 'data', 'outputs', 'external']
for folder in required:
    path = PROJECT_ROOT / folder
    print(f'{folder:10s}', 'OK' if path.exists() else 'missing')

## Step 3: Install core packages if needed

Notebook 00 installs only the lightweight core dependencies. NER packages are installed in Notebook 01 because users first need to choose which NER backend they want to run.

In [ ]:
RUN_CORE_INSTALL = False  # change to True if any core package is missing below

if RUN_CORE_INSTALL:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(PROJECT_ROOT / "requirements-core.txt"),
    ])
else:
    print("Core install skipped. Set RUN_CORE_INSTALL = True if a core package is missing.")

## Step 4: Check required core packages

These packages are needed for the basic workshop path. If one is missing, run the install cell above, then restart the kernel and rerun this notebook.

In [ ]:
core_packages = {'pandas': 'pandas', 'requests': 'requests', 'folium': 'folium', 'python-dotenv': 'dotenv'}
for label, import_name in core_packages.items():
    status = 'available' if importlib.util.find_spec(import_name) else 'missing - run the core install cell above'
    print(f'{label:16s} {status}')

## Step 5: Load project helpers after core packages are ready

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RESULTS_DIR, GEONAMES_BASE_URL, PHOTON_BASE_URL, REQUEST_TIMEOUT
from src.data_utils import load_sample_texts, save_dataframe
import requests

## Step 6: Load sample texts

In [ ]:
texts = load_sample_texts()
texts

## Step 7: Save sample texts for later notebooks

In [ ]:
out = save_dataframe(texts, RESULTS_DIR / 'sample_texts.csv')
print('Saved:', out)

## Step 8: Check internal service reachability

These services may require the DLR internal network or VPN. You can still continue with sample outputs.

In [ ]:
def check_service(url, params):
    try:
        r = requests.get(url, params=params, timeout=min(REQUEST_TIMEOUT, 3))
        return r.status_code, r.url
    except Exception as exc:
        return 'unreachable', str(exc)

print('GeoNames:', check_service(GEONAMES_BASE_URL, {'location': 'Berlin'}))
print('Photon:', check_service(PHOTON_BASE_URL, {'q': 'Berlin', 'limit': 1}))

## Common issues

- A DLR service may be unreachable outside the internal network or VPN.
- If imports from `src` fail, start Jupyter from the project root.